# Deploy a learned policy *safely*: dual-path MPC + AI with guaranteed fallback

Modern robots want two kinds of controller at once:

- a **model-based controller** (MPC/LQR) — predictable, respects constraints,
  but conservative;
- a **learned policy** (trained from data) — smoother and smarter in normal
  operation, but occasionally unpredictable, and it can stall.

You want the AI's skill *and* the model-based controller's safety. This tutorial
compiles **both into one binary** for the robot's chip, with a referee — the
**arbiter** — between them. Every cycle the arbiter lets the policy adjust the
MPC's command, but **instantly falls back to the constraint-respecting MPC** if
the policy is late (a timeout) or the robot drifts out of its safe box, and
**clamps** the final command into the actuator limits no matter what.

```
            ┌─ acados MPC (high rate, hard real-time) ─┐
   state ──►┤                                          ├─► arbiter ─► clamp ─► command
            └─ learned policy (lower rate)  ───────────┘   (fallback to MPC on
                                                            timeout / envelope breach)
```

The promise: **the AI can help, but it can never put the robot into an unsafe
state** — the safe controller is always one cycle away. And it's all attested, so
you can prove exactly which controller + which policy + which limits you shipped.

In [1]:
import os, platform, ctypes
from pathlib import Path
def _preload_acados():
    root = os.environ.get("ACADOS_SOURCE_DIR")
    if not root or platform.system() != "Darwin": return
    for n in ("libblasfeo.dylib","libhpipm.dylib","libqpOASES_e.dylib","libacados.dylib"):
        p = Path(root)/"lib"/n
        if p.exists():
            try: ctypes.CDLL(str(p))
            except OSError: pass
_preload_acados()
import numpy as np, jax, jax.numpy as jnp
print("acados", "set" if os.environ.get("ACADOS_SOURCE_DIR") else "MISSING (build cells fail)")

acados set


## Step 1 — The model-based controller (MPC)

The safe baseline: the Cartpole LQR controller, built exactly as in tutorial #1.

In [2]:
import tempfile
import jaxterity.zoo as zoo
from jaxterity.zoo.cartpole import reduced_params
from jaxility.lowering import translate
from jaxility.templates import lqr
from jaxility.builder import build_for_target
from jaxility.targets import current_host_target

NX, NU = 4, 1
INITIAL_STATE = (0.3, 0.0, 0.0, 0.0)
PLANT_DT = 1.0/20
robot = zoo.load("cartpole").with_provenance(("dual-tut","v0","tel"), calibrated=True)
p = reduced_params(robot)
def dyn(state, control):
    g, mp, mc, L = p["g"], p["mp"], p["mc"], p["L"]
    th, xd, thd = state[1], state[2], state[3]
    s, c = jnp.sin(th), jnp.cos(th); den = mc + mp*s*s
    return jnp.array([xd, thd, (control[0]+mp*s*(L*thd**2+g*c))/den,
        (-control[0]*c-mp*L*thd**2*c*s-(mc+mp)*g*s)/(L*den)])
cf = translate(dyn, in_shapes=((NX,),(NU,)), name="dual_cartpole")
spec = lqr(cf, Q=(10.,10.,1.,1.), R=(0.1,), initial_state=INITIAL_STATE,
           input_bounds=((-20.,),(20.,)), name="dual_cartpole", horizon_steps=20, time_horizon_s=1.0)
bundle = build_for_target(dynamics=cf, spec=spec, target=current_host_target(),
    source_attestation_handle=bytes.fromhex(robot.attestation_handle),
    work_dir=Path(tempfile.mkdtemp()))
print("MPC controller built — manifest verifies:",
      __import__("jaxility").manifest.verify_manifest(bundle.manifest).ok)

MPC controller built — manifest verifies: True


## Step 2 — The learned policy (the AI)

A small MLP `state → residual force`. In practice this is trained (e.g. with RL)
and exported through `jaxility.policy` (JAX → ONNX → LiteRT, tutorial-worthy in its
own right); here we use a tiny flax net so we can focus on the **safe deployment**.
`MLPPolicy.from_flax` captures its weights, and the same forward pass is emitted
into C — so what runs on the chip is exactly this function.

In [3]:
import flax.linen as nn
from jaxility.compose import MLPPolicy

class Pol(nn.Module):
    @nn.compact
    def __call__(self, x):
        return nn.Dense(1)(nn.tanh(nn.Dense(8)(x)))

params = Pol().init(jax.random.PRNGKey(3), jnp.zeros((4,)))
policy = MLPPolicy.from_flax(params, input_dim=4)

x_demo = np.array([0.3, 0.05, 0.0, 0.0])
bundle.solver.reset(); bundle.solver.set(0,"lbx",x_demo); bundle.solver.set(0,"ubx",x_demo)
bundle.solver.solve(); u_mpc = float(bundle.solver.get(0,"u")[0])
u_pol = float(policy.forward(x_demo)[0])
print(f"at state {x_demo.tolist()}:")
print(f"  MPC command      u_mpc   = {u_mpc:+.4f} N")
print(f"  policy residual  u_policy= {u_pol:+.4f} N")
print(f"  combined (dual)  u       = {u_mpc + u_pol:+.4f} N   (clamped into [-20, 20])")

at state [0.3, 0.05, 0.0, 0.0]:
  MPC command      u_mpc   = -4.6972 N
  policy residual  u_policy= +0.1177 N
  combined (dual)  u       = -4.5796 N   (clamped into [-20, 20])


## Step 3 — The composition contract & the arbiter

The safety surface is **declarative**, not buried in code: a `CompositionPlan`
names the two rates, the `SafetyEnvelope` (the state box and the actuator box), and
the fallback policy. `arbitrate(...)` is the reference one-step decision the
on-target C mirrors. Let's watch it make all three decisions.

In [4]:
from jaxility.compose import CompositionPlan, SafetyEnvelope, arbitrate

env = SafetyEnvelope(name="cartpole",
    state_lower=(-5.,-3.,-20.,-20.), state_upper=(5.,3.,20.,20.),
    input_lower=(-20.,), input_upper=(20.,))
plan = CompositionPlan(name="cartpole-dual", mpc_period_ns=1_000_000,
    policy_period_ns=20_000_000, safety_envelope=env)   # MPC 1 kHz, policy 50 Hz

x = np.array([0.3, 0.05, 0.0, 0.0])
u_mpc = np.array([u_mpc]); u_pol = policy.forward(x)

# (a) normal: both paths combine
r = arbitrate(plan, mpc_control=u_mpc, policy_action=u_pol, state=x, policy_timed_out=False)
print(f"(a) normal         -> path={r.path:13s} command={r.command[0]:+.3f}  reason={r.fallback_reason}")
# (b) the policy missed its deadline -> fall back to MPC
r = arbitrate(plan, mpc_control=u_mpc, policy_action=u_pol, state=x, policy_timed_out=True)
print(f"(b) policy timeout -> path={r.path:13s} command={r.command[0]:+.3f}  reason={r.fallback_reason}")
# (c) the state left the safe box -> fall back to MPC
x_bad = np.array([9.0, 0.05, 0.0, 0.0])   # cart far outside [-5, 5]
r = arbitrate(plan, mpc_control=u_mpc, policy_action=u_pol, state=x_bad, policy_timed_out=False)
print(f"(c) envelope breach-> path={r.path:13s} command={r.command[0]:+.3f}  reason={r.fallback_reason}")
print("\n-> in (b) and (c) the learned policy is dropped; the safe MPC command stands.")

(a) normal         -> path=dual          command=-4.580  reason=None
(b) policy timeout -> path=mpc_fallback  command=-4.697  reason=timeout
(c) envelope breach-> path=mpc_fallback  command=-4.697  reason=envelope

-> in (b) and (c) the learned policy is dropped; the safe MPC command stands.


## Step 4 — Compile the dual-path binary, and watch the fallback fire

Now jaxility emits the whole thing — MPC + the policy's forward pass + the arbiter
+ the clamp — as one native binary. We run it two ways and compare on real output:
with the policy **active**, and with a **forced timeout** (the deployed binary
accepts a `fallback_from_step` argument so the fallback is testable).

In [5]:
import subprocess
from jaxility.compose import generate_dual_path_hil_source
from jaxility.hil import build_controller_hil_binary, CARTPOLE_LQR_SCHEMA, parse_trace

src = generate_dual_path_hil_source(model_name="dual_cartpole", nx=NX, nu=NU,
        policy=policy, plan=plan, initial_state=INITIAL_STATE)
gen = bundle.shared_library_path.parent
exe = build_controller_hil_binary(generated_code_dir=gen, model_name="dual_cartpole",
        source=src, out_path=gen/"dual_cartpole_hil")

def run(fb_from):
    out = subprocess.run([str(exe), "40", "0", str(fb_from)], capture_output=True, text=True).stdout
    return parse_trace(out, CARTPOLE_LQR_SCHEMA, n_steps=40)

dual = run(-1)      # policy active every cycle
fallback = run(0)   # forced policy timeout from step 0 -> MPC only
du = dual["actuator_torque"][:, 0]
fu = fallback["actuator_torque"][:, 0]
print("first 5 commands, dual (policy active):    ", np.round(du[:5], 3).tolist())
print("first 5 commands, forced fallback (MPC):   ", np.round(fu[:5], 3).tolist())
print(f"\npolicy genuinely changes the command: max |dual - fallback| = {np.abs(du-fu).max():.3f} N")
print("forced fallback == the constraint-respecting MPC command (safe baseline).")

first 5 commands, dual (policy active):     [-5.412, -1.751, 0.112, 0.873, 1.005]
first 5 commands, forced fallback (MPC):    [-5.525, -2.021, -0.03, 0.893, 1.145]

policy genuinely changes the command: max |dual - fallback| = 0.269 N
forced fallback == the constraint-respecting MPC command (safe baseline).


**That is the safety guarantee, on real generated code.** With the policy
active, the command reflects its contribution. The instant we force a timeout, the
binary drops to the clamped MPC command — no AItorque leaks through. The same is
true for an envelope breach (decision (c) above). And because this is the *same*
binary the HIL harness validates step-locked against the host arbiter (tutorial #2),
the fallback isn't a hope — it's verified.

## Step 5 — Attestation covers *both* artifacts

A dual-path deployment ships two artifacts — the MPC controller and the policy —
under a safety envelope. The attestation binds **all three** into one verifiable
hash. Change any of them — recalibrate the robot, retrain the policy, loosen the
envelope — and the attestation moves. That's the audit trail.

In [6]:
import blake3
from jaxility.compose import attest_dual_path

# the policy's content = its weights, serialized deterministically
def policy_bytes(pol):
    return b"".join(np.asarray(l.weight, np.float64).tobytes()
                    + np.asarray(l.bias, np.float64).tobytes() for l in pol.layers)

att = attest_dual_path(controller_manifest=bundle.manifest,
                       policy_artifact_bytes=policy_bytes(policy), plan=plan)
print("dual-path attestation hash:", att.content_hash().hex()[:32], "…")
print("  binds: controller manifest + policy weights + composition plan\n")

# retrain the policy (different seed) -> the attestation must move
policy2 = MLPPolicy.from_flax(Pol().init(jax.random.PRNGKey(7), jnp.zeros((4,))), input_dim=4)
att2 = attest_dual_path(controller_manifest=bundle.manifest,
                        policy_artifact_bytes=policy_bytes(policy2), plan=plan)
# loosen the envelope -> also moves
plan3 = CompositionPlan(name="cartpole-dual", mpc_period_ns=1_000_000, policy_period_ns=20_000_000,
    safety_envelope=SafetyEnvelope(name="loose", state_lower=(-9.,-9.,-99.,-99.),
        state_upper=(9.,9.,99.,99.), input_lower=(-20.,), input_upper=(20.,)))
att3 = attest_dual_path(controller_manifest=bundle.manifest,
                        policy_artifact_bytes=policy_bytes(policy), plan=plan3)
print("retrain the policy   -> attestation moves:", att.content_hash() != att2.content_hash())
print("loosen the envelope  -> attestation moves:", att.content_hash() != att3.content_hash())

dual-path attestation hash: cf5c51bdd784687f7256b8b84b8d2e6b …
  binds: controller manifest + policy weights + composition plan

retrain the policy   -> attestation moves: True
loosen the envelope  -> attestation moves: True


## Step 6 — Is safety free? The real-time cost

Adding the AI + the referee to every MPC cycle had better not blow the control
period. We time the **whole composed cycle** (solve + policy + arbiter + clamp) and
compare it to the bare MPC.

In [7]:
from jaxility.compose import generate_dual_path_bench_source
from jaxility.bench import generate_controller_bench_source, run_controller_bench
from jaxility.hil import LocalRunner

def bench(src, tag):
    e = build_controller_hil_binary(generated_code_dir=gen, model_name="dual_cartpole",
            source=src, out_path=gen/f"dual_cartpole_{tag}")
    return run_controller_bench(LocalRunner(e), robot="cartpole", target_family="cortex-a76",
            target_name="host", n_cycles=3000, n_warmup=100)

sp = bench(generate_controller_bench_source(model_name="dual_cartpole", nx=NX, nu=NU,
        initial_state=INITIAL_STATE, n_warmup=100), "bsp")
dp = bench(generate_dual_path_bench_source(model_name="dual_cartpole", nx=NX, nu=NU,
        policy=policy, plan=plan, initial_state=INITIAL_STATE, n_warmup=100), "bdp")
print(f"host single-path MPC : mean {sp.solve.mean_ns/1e3:5.1f} us  p99 {sp.solve.p99_ns/1e3:5.1f}")
print(f"host dual-path       : mean {dp.solve.mean_ns/1e3:5.1f} us  p99 {dp.solve.p99_ns/1e3:5.1f}")
print(f"overhead of the AI + arbiter: {(dp.solve.mean_ns-sp.solve.mean_ns)/1e3:+.2f} us "
      f"({100*(dp.solve.mean_ns/sp.solve.mean_ns-1):+.1f}%)")

host single-path MPC : mean  30.0 us  p99  65.0
host dual-path       : mean  29.7 us  p99  64.0
overhead of the AI + arbiter: -0.26 us (-0.9%)


On the host that's ~30 µs; the number that matters is **real silicon**. The
cell below cross-compiles *both* binaries onto a tethered **Raspberry Pi 5**
(Cortex-A76) and times them there. (Set `JAXILITY_HIL_SSH_HOST`; if unset, the
host comparison above still makes the point.)

In [8]:
import subprocess
from jaxility.hil import build_controller_on_target

host = os.environ.get("JAXILITY_HIL_SSH_HOST")
if not host:
    print("JAXILITY_HIL_SSH_HOST unset — skipping the on-silicon timing (host result above stands).")
else:
  try:
    acados = os.environ.get("JAXILITY_HIL_ACADOS") or subprocess.run(
        ["ssh","-o","BatchMode=yes","-o","ConnectTimeout=20",host,"echo $HOME/acados"],
        capture_output=True, text=True, timeout=30).stdout.strip()
    def pibench(src, tag):
        r = build_controller_on_target(host=host, generated_code_dir=gen, model_name="dual_cartpole",
            source=src, source_name=f"dual_{tag}_main.c", remote_dir=f"/tmp/jx-dual-{tag}",
            remote_acados=acados)
        return run_controller_bench(r, robot="cartpole", target_family="cortex-a76",
            target_name="pi5", n_cycles=5000, n_warmup=100)
    psp = pibench(generate_controller_bench_source(model_name="dual_cartpole", nx=NX, nu=NU,
            initial_state=INITIAL_STATE, n_warmup=100), "sp")
    pdp = pibench(generate_dual_path_bench_source(model_name="dual_cartpole", nx=NX, nu=NU,
            policy=policy, plan=plan, initial_state=INITIAL_STATE, n_warmup=100), "dp")
    print(f"on-Pi single-path MPC : mean {psp.solve.mean_ns/1e3:5.1f} us  p99 {psp.solve.p99_ns/1e3:5.1f}  1kHz={psp.meets_1khz}")
    print(f"on-Pi dual-path       : mean {pdp.solve.mean_ns/1e3:5.1f} us  p99 {pdp.solve.p99_ns/1e3:5.1f}  1kHz={pdp.meets_1khz}")
    print(f"on-Pi overhead of the AI + arbiter: {(pdp.solve.mean_ns-psp.solve.mean_ns)/1e3:+.2f} us")
    print("\n-> on real silicon, the learned path + arbiter is within run-to-run noise:")
    print("   composing the AI controller and the safety referee onto the MPC is")
    print("   essentially FREE — the dual-path meets 1 kHz with the same margin as the MPC.")
  except Exception as e:
    print(f"on-silicon timing skipped ({type(e).__name__}: {e}); the host result above stands.")

on-Pi single-path MPC : mean  99.0 us  p99 135.9  1kHz=True
on-Pi dual-path       : mean  99.4 us  p99 135.8  1kHz=True
on-Pi overhead of the AI + arbiter: +0.36 us

-> on real silicon, the learned path + arbiter is within run-to-run noise:
   composing the AI controller and the safety referee onto the MPC is
   essentially FREE — the dual-path meets 1 kHz with the same margin as the MPC.


## Recap

You deployed a **learned policy on top of a model-based controller, safely**, into
one binary for the robot's chip:

- the arbiter combines the two and **falls back to the constraint-respecting MPC**
  on a policy timeout or an envelope breach — demonstrated on the generated code;
- the final command is always **clamped** into the actuator limits;
- the deployment is **attested over both artifacts + the safety envelope** —
  retrain or re-envelope and the attestation moves;
- and it's **essentially free** at 1 kHz on real silicon.

This is "AI you can certify": the learned policy improves behaviour in the common
case, and the model-based controller guarantees safety in every case — with a
cryptographic record of exactly what you shipped.